# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra-Jahangir/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Lane: Structured Content Archetype Clustering.

I'm choosing this lane because the starter dataset has a rich mix of numeric signals per page (impressions, clicks, sessions, CTR, position, word count, age, engagement rate, scroll rate) plus categorical context (content type, intent, competition level). Rather than starting with a single prediction target, I want to first understand what natural "types" of pages exist in this inventory — e.g. high-traffic stable pages, thin low-engagement pages, new pages still building visibility. Clustering lets me discover these groups from the data itself, which can then inform which archetypes deserve which kind of action (protect, improve, rewrite, prune, monitor). This also sets up later weeks well, since I can revisit these clusters once I define specific labels for prediction.


In [5]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Rows, columns:", df.shape)
print("Numeric-looking columns available for clustering:",
      df.select_dtypes(include="number").shape[1])

Rows, columns: (30000, 44)
Numeric-looking columns available for clustering: 30


## 2. The question: decision, action, cost of a wrong call

Unit of analysis: one content page (content_id).

The question: What performance archetypes exist across FlyRank's content inventory, based on safe engagement and visibility metrics?

The decision: Instead of treating every page identically, which group does a given page belong to, so the content team can apply a consistent strategy per group rather than reviewing pages one by one from scratch?

Who acts, and how: A content strategist uses the cluster label to decide the general treatment for a page. e.g. "champion" pages get protected/monitored only, "stale visible" pages get prioritized for refresh, "weak/no-demand" pages get considered for pruning.

Cost of a wrong call: If a page is misclassified into the wrong archetype, the team could waste effort refreshing a page that's actually fine (false positive), or leave a genuinely declining page in a "safe" cluster and it keeps losing traffic unnoticed (false negative). Because clusters guide strategy, not one specific action, wrong assignment mostly costs misallocated attention rather than a single bad edit.

Why ML/clustering, not a simple rule: With 30,000+ pages and 6+ numeric signals interacting at once, no simple threshold rule can capture the combinations that define a "type" of page. Clustering lets the data reveal these groupings rather than a person guessing categories in advance.

In [6]:
print("Unique content_id count:", df["content_id"].nunique())
print("Total rows:", len(df))
# If these match, each row is a distinct page (no duplicates to worry about)

Unique content_id count: 30000
Total rows: 30000


## 3. Quick look at the data (2-3 real numbers)

This dataset has enough scale and variance to support meaningful clustering. Across 30,000 pages, impressions_90d ranges from 1 to 517,715, with a median of just 731 versus a mean of ~5,200 — a heavily skewed distribution suggesting a small set of very high-traffic "champion" pages alongside a long tail of low-traffic ones, exactly the separation clustering should surface. Content type also splits unevenly across three categories (27,207 keyword articles, 2,096 feedly articles, 697 comparison articles), giving categorical structure to profile clusters against. Engagement is similarly polarized: engagement_rate has a median of 0 but a max of 100, hinting at a distinct "engagement-problem" archetype worth isolating.

In [7]:
# 1. Overall scale
print("Total pages:", len(df))

# 2. Spread in impressions_90d : real variance is what makes clustering meaningful
print("\nimpressions_90d spread:")
print(df["impressions_90d"].describe())

# 3. Content type mix :categorical diversity that could map to distinct archetypes
print("\ncontent_type counts:")
print(df["content_type"].value_counts())

# 4. Engagement rate spread : another axis pages could cluster along
print("\nengagement_rate spread:")
print(df["engagement_rate"].describe())

Total pages: 30000

impressions_90d spread:
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64

content_type counts:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64

engagement_rate spread:
count    30000.000000
mean         2.534520
std          8.310096
min          0.000000
25%          0.000000
50%          0.000000
75%          1.350000
max        100.000000
Name: engagement_rate, dtype: float64


## 4. Careful words: what I can and can't claim

This analysis is exploratory and unsupervised, there is no "correct" set of clusters to validate against, only clusters that are internally consistent and interpretable. I can say clusters are "associated with" certain metric patterns, and describe them as archetypes, but I cannot claim these groups represent true causal categories, nor that assigning a page to a cluster and acting on it will guarantee any specific outcome. I will not call this "semantic" clustering, since the data contains no article text, only structured metrics and metadata. Cluster boundaries depend on the features and scaling choices I make, so results are a lens, not ground truth

In [8]:
# Sanity check: confirm this is unsupervised - no target column being used
# (trend_direction/trend_pct exist but are NOT inputs to clustering, only optional
# post-hoc profiling of what a cluster tends to look like)
print("Reminder: clustering will use only observed metric columns, not trend_direction/trend_pct as inputs.")
print(df[["trend_direction"]].value_counts())

Reminder: clustering will use only observed metric columns, not trend_direction/trend_pct as inputs.
trend_direction
down               16262
stable              5962
up                  4388
new                 2236
flat                1152
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.